
# SESSÃO 11 — CONSTRUÇÃO DA MATRIZ DE ATRIBUTOS V4

**Projeto:** Classificação supervisionada de eventos indesejados — 3W 2.0.0 (C1).

## Objetivo

Construir uma matriz de atributos estatísticos a partir das
janelas temporais dos arquivos de treinamento da V4 ajustada.

## Fundamentação

A Sessão 10 demonstrou que a utilização de sensores alternativos
aumenta a cobertura dos poços e das janelas disponíveis.

Nesta sessão, a estratégia candidata será exigir pelo menos
dois dos cinco sensores completos em cada janela.

## Configuração experimental

- Particionamento: V4 ajustada.
- Conjunto utilizado: somente treinamento.
- Janela temporal: 100 amostras.
- Deslocamento: 50 amostras.
- Sensores: P-TPT, T-TPT, P-ANULAR, P-JUS-CKGL e P-MON-CKP.
- Rótulos temporais: preservar os códigos originais.
- Sensores ausentes: representar explicitamente sua disponibilidade.
- Validação e teste: não utilizados nesta construção inicial.

## Resultado esperado

Uma matriz com atributos estatísticos, rótulos temporais,
identificação dos arquivos e poços, acompanhada de
verificações de integridade e distribuição dos dados.


In [ ]:

# SESSAO 11 - TAREFA 01
# Recuperar o protocolo V4 e a auditoria da Sessao 10

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Procurar os arquivos ja existentes
raiz = Path("/content/drive/MyDrive")

manifestos = list(
    raiz.rglob("*V4_Instancias_CANDIDATA_AJUSTADA.csv")
)

manifestos = [
    p for p in manifestos
    if p.parent.name == "PROTOCOLO_V4_COMPARTILHADO"
]

auditorias = list(
    raiz.rglob("10_Auditoria_535_Arquivos_V4.csv")
)

assert len(manifestos) == 1, manifestos
assert len(auditorias) == 1, auditorias

# Carregar os dados sem alterá-los
v4 = pd.read_csv(manifestos[0])
aud = pd.read_csv(auditorias[0])

# Selecionar somente os arquivos do treinamento
treino = v4[v4["Conjunto"] == "Treino"].copy()

# Conferir consistencia
assert len(treino) == 535
assert treino["Poco"].nunique() == 28
assert len(aud) == 535
assert aud["Status"].eq("OK").all()

assert set(treino["Arquivo"]) == set(aud["Arquivo"])

# Configuracoes candidatas para a matriz
SENSORES = [
    "P-TPT",
    "T-TPT",
    "P-ANULAR",
    "P-JUS-CKGL",
    "P-MON-CKP"
]

TAMANHO = 100
PASSO = 50
MIN_SENSORES = 2

print("PROTOCOLO V4 CARREGADO")
print("Arquivos de treinamento:", len(treino))
print("Pocos de treinamento:", treino["Poco"].nunique())

print("\nCONFIGURACAO:")
print("Janela:", TAMANHO)
print("Passo:", PASSO)
print("Minimo de sensores:", MIN_SENSORES)

print("\nJANELAS CANDIDATAS:")
print(int(aud["Janelas_2_ou_mais"].sum()))

print("\nTAREFA 01 EXECUTADA")


Mounted at /content/drive
PROTOCOLO V4 CARREGADO
Arquivos de treinamento: 535
Pocos de treinamento: 28

CONFIGURACAO:
Janela: 100
Passo: 50
Minimo de sensores: 2

JANELAS CANDIDATAS:
321929

TAREFA 01 EXECUTADA


In [ ]:

# SESSAO 11 - TAREFA 02
# Extracao piloto de atributos estatisticos

import io
import requests
import numpy as np
import pandas as pd

# Arquivo piloto pertencente ao treinamento V4
nome = "WELL-00002_20131104004101.parquet"

item = treino.loc[
    treino["Arquivo"] == nome
].iloc[0]

url = (
    "https://raw.githubusercontent.com/petrobras/3W/"
    f"main/dataset/{item['Categoria']}/{nome}"
)

resposta = requests.get(url, timeout=120)
resposta.raise_for_status()

dados = pd.read_parquet(io.BytesIO(resposta.content))

# Selecionar sensores e rotulos
sinais = dados.reindex(columns=SENSORES).apply(
    pd.to_numeric, errors="coerce"
).to_numpy(dtype=float)

rotulos = pd.to_numeric(
    dados["class"], errors="coerce"
).to_numpy(dtype=float)

rotulos_validos = set(range(10)) | set(range(101, 110))

# Preparar o calculo da inclinacao
tempo = np.arange(TAMANHO, dtype=float)
tempo_centrado = tempo - tempo.mean()
denominador = np.sum(tempo_centrado ** 2)

linhas = []

for inicio in range(0, len(dados) - TAMANHO + 1, PASSO):

    fim = inicio + TAMANHO

    r = rotulos[inicio:fim]

    # Aceitar apenas rotulos validos e constantes
    if (
        not np.isfinite(r).all()
        or r[0] not in rotulos_validos
        or not np.all(r == r[0])
    ):
        continue

    # Verificar continuidade temporal
    delta = (
        np.diff(dados.index.asi8[inicio:fim]) / 1e9
    )

    if not np.isclose(delta, 1, atol=0.1).all():
        continue

    bloco = sinais[inicio:fim]

    # Sensor completo = 100 valores finitos na janela
    completos = np.isfinite(bloco).all(axis=0)

    if completos.sum() < MIN_SENSORES:
        continue

    linha = {
        "Arquivo": nome,
        "Poco": item["Poco"],
        "Categoria_arquivo": int(item["Categoria"]),
        "Rotulo_original": int(r[0]),
        "Inicio": str(dados.index[inicio]),
        "Fim": str(dados.index[fim - 1]),
        "N_sensores": int(completos.sum())
    }

    for j, sensor in enumerate(SENSORES):

        linha[f"{sensor}_disponivel"] = int(completos[j])

        atributos = [
            "media", "desvio", "inclinacao",
            "p10", "p90"
        ]

        if completos[j]:
            valores = bloco[:, j]

            estatisticas = [
                np.mean(valores),
                np.std(valores),
                np.dot(
                    tempo_centrado,
                    valores - np.mean(valores)
                ) / denominador,
                np.percentile(valores, 10),
                np.percentile(valores, 90)
            ]
        else:
            estatisticas = [np.nan] * 5

        for atributo, valor in zip(
            atributos, estatisticas
        ):
            linha[f"{sensor}_{atributo}"] = valor

    linhas.append(linha)

matriz_piloto = pd.DataFrame(linhas)

# Comparar a quantidade com a auditoria anterior
esperado = int(
    aud.loc[
        aud["Arquivo"] == nome,
        "Janelas_2_ou_mais"
    ].iloc[0]
)

assert len(matriz_piloto) == esperado

print("MATRIZ PILOTO CONSTRUIDA")
print("Janelas:", len(matriz_piloto))
print("Colunas totais:", matriz_piloto.shape[1])
print("Atributos estatisticos:", 25)
print("Indicadores de disponibilidade:", 5)

print("\nROTULOS TEMPORAIS ORIGINAIS:")
display(
    matriz_piloto["Rotulo_original"]
    .value_counts()
    .sort_index()
)

print("\nEXEMPLO DAS PRIMEIRAS JANELAS:")
display(
    matriz_piloto[
        [
            "Poco",
            "Rotulo_original",
            "N_sensores",
            "P-TPT_media",
            "P-TPT_desvio",
            "P-JUS-CKGL_disponivel"
        ]
    ].head()
)

print("\nTAREFA 02 EXECUTADA")


MATRIZ PILOTO CONSTRUIDA
Janelas: 177
Colunas totais: 37
Atributos estatisticos: 25
Indicadores de disponibilidade: 5

ROTULOS TEMPORAIS ORIGINAIS:


,count
Rotulo_original,
0,74
2,2
102,101



EXEMPLO DAS PRIMEIRAS JANELAS:


,Poco,Rotulo_original,N_sensores,P-TPT_media,P-TPT_desvio,P-JUS-CKGL_disponivel
0,WELL-00002,0,3,18852350.0,0.0,0
1,WELL-00002,0,3,18852350.0,0.0,0
2,WELL-00002,0,3,18852350.0,0.0,0
3,WELL-00002,0,3,18852350.0,0.0,0
4,WELL-00002,0,3,18852350.0,0.0,0



TAREFA 02 EXECUTADA


In [ ]:

# SESSAO 11 - TAREFA 03
# Verificar variabilidade dos sensores na matriz piloto

resultados_qualidade = []

for sensor in SENSORES:

    disponivel = matriz_piloto[
        f"{sensor}_disponivel"
    ] == 1

    desvio = matriz_piloto[
        f"{sensor}_desvio"
    ]

    constante = disponivel & np.isclose(
        desvio, 0, atol=1e-10
    )

    resultados_qualidade.append({
        "Sensor": sensor,
        "Janelas_completas": int(disponivel.sum()),
        "Janelas_constantes": int(constante.sum()),
        "Percentual_constantes": round(
            100 * constante.sum() / max(disponivel.sum(), 1),
            2
        )
    })

qualidade = pd.DataFrame(resultados_qualidade)

print("QUALIDADE DOS SENSORES — WELL-00002:")
display(qualidade)

print("\nDISTRIBUICAO DE SENSORES COMPLETOS:")
display(
    matriz_piloto["N_sensores"]
    .value_counts()
    .sort_index()
)

print("\nTAREFA 03 EXECUTADA")


QUALIDADE DOS SENSORES — WELL-00002:


,Sensor,Janelas_completas,Janelas_constantes,Percentual_constantes
0,P-TPT,177,72,40.68
1,T-TPT,177,10,5.65
2,P-ANULAR,177,63,35.59
3,P-JUS-CKGL,0,0,0.00
4,P-MON-CKP,0,0,0.00



DISTRIBUICAO DE SENSORES COMPLETOS:


,count
N_sensores,
3,177



TAREFA 03 EXECUTADA


In [ ]:

# SESSAO 11 - TAREFA 04
# Teste de extracao com sensores originais ausentes

import io
import requests

# Selecionar um arquivo do WELL-00023
item23 = (
    treino[treino["Poco"] == "WELL-00023"]
    .sort_values("Arquivo")
    .iloc[0]
)

url23 = (
    "https://raw.githubusercontent.com/petrobras/3W/"
    f"main/dataset/{item23['Categoria']}/{item23['Arquivo']}"
)

resposta = requests.get(url23, timeout=180)
resposta.raise_for_status()

dados23 = pd.read_parquet(io.BytesIO(resposta.content))

sinais23 = dados23.reindex(
    columns=SENSORES
).apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

rotulos23 = pd.to_numeric(
    dados23["class"], errors="coerce"
).to_numpy(dtype=float)

tempo = np.arange(TAMANHO, dtype=float)
tc = tempo - tempo.mean()
den = np.sum(tc**2)

atributos = [
    "media", "desvio", "inclinacao", "p10", "p90"
]

linhas23 = []

for inicio in range(0, len(dados23) - TAMANHO + 1, PASSO):

    fim = inicio + TAMANHO
    r = rotulos23[inicio:fim]

    if (
        not np.isfinite(r).all()
        or r[0] not in rotulos_validos
        or not np.all(r == r[0])
    ):
        continue

    delta = np.diff(
        dados23.index.asi8[inicio:fim]
    ) / 1e9

    if not np.isclose(delta, 1, atol=0.1).all():
        continue

    bloco = sinais23[inicio:fim]
    completos = np.isfinite(bloco).all(axis=0)

    if completos.sum() < MIN_SENSORES:
        continue

    linha = {
        "Arquivo": item23["Arquivo"],
        "Poco": item23["Poco"],
        "Categoria_arquivo": int(item23["Categoria"]),
        "Rotulo_original": int(r[0]),
        "Inicio": str(dados23.index[inicio]),
        "Fim": str(dados23.index[fim - 1]),
        "N_sensores": int(completos.sum())
    }

    for j, sensor in enumerate(SENSORES):

        linha[f"{sensor}_disponivel"] = int(completos[j])

        if completos[j]:
            valores = bloco[:, j]
            media = np.mean(valores)

            estatisticas = [
                media,
                np.std(valores),
                np.dot(tc, valores - media) / den,
                np.percentile(valores, 10),
                np.percentile(valores, 90)
            ]
        else:
            estatisticas = [np.nan] * 5

        for atributo, valor in zip(
            atributos, estatisticas
        ):
            linha[f"{sensor}_{atributo}"] = valor

    linhas23.append(linha)

    # Somente as primeiras 30 janelas elegiveis
    if len(linhas23) == 30:
        break

matriz23 = pd.DataFrame(linhas23)

# Conferir a estrutura da matriz
assert len(matriz23) == 30
assert list(matriz23.columns) == list(matriz_piloto.columns)

print("MATRIZ PILOTO WELL-00023:")
print("Janelas:", len(matriz23))
print("Colunas:", matriz23.shape[1])

print("\nDISPONIBILIDADE DOS SENSORES:")
display(
    matriz23[
        [f"{s}_disponivel" for s in SENSORES]
    ].sum().to_frame("Janelas_com_sensor")
)

print("\nEXEMPLO DOS ATRIBUTOS:")
display(
    matriz23[
        [
            "Poco",
            "Rotulo_original",
            "N_sensores",
            "P-TPT_media",
            "P-JUS-CKGL_media",
            "P-MON-CKP_media"
        ]
    ].head()
)

print("\nTAREFA 04 EXECUTADA")


MATRIZ PILOTO WELL-00023:
Janelas: 30
Colunas: 37

DISPONIBILIDADE DOS SENSORES:


,Janelas_com_sensor
P-TPT_disponivel,0
T-TPT_disponivel,0
P-ANULAR_disponivel,0
P-JUS-CKGL_disponivel,30
P-MON-CKP_disponivel,30



EXEMPLO DOS ATRIBUTOS:


,Poco,Rotulo_original,N_sensores,P-TPT_media,P-JUS-CKGL_media,P-MON-CKP_media
0,WELL-00023,0,2,NaN,16235633.8,4170068.38
1,WELL-00023,0,2,NaN,16233348.7,4160444.27
2,WELL-00023,0,2,NaN,16232192.1,4158509.37
3,WELL-00023,0,2,NaN,16231980.0,4171406.25
4,WELL-00023,0,2,NaN,16230862.8,4171926.70



TAREFA 04 EXECUTADA


In [ ]:

# SESSAO 11 - TAREFA 05
# Extracao e salvamento da matriz V4 por arquivo

import io
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view

# Usar a pasta ja verificada na Tarefa 01
pasta_saida = auditorias[0].parent / "11_Matriz_Atributos_V4"
pasta_saida.mkdir(exist_ok=True)

registro = pasta_saida / "11_Progresso_Extracao.csv"

# Recuperar o progresso anterior, se existir
if registro.exists():
    progresso = pd.read_csv(registro)
else:
    progresso = pd.DataFrame(
        columns=["Arquivo", "Poco", "Janelas", "Status"]
    )

concluidos = set(
    progresso.loc[progresso["Status"] == "OK", "Arquivo"]
)

# Percorrer um arquivo por vez, inicialmente em lote de 30
ordem = treino.copy()
ordem["Ordem"] = ordem.groupby("Poco").cumcount()
ordem = ordem.sort_values(["Ordem", "Poco", "Arquivo"])

lote = ordem[
    ~ordem["Arquivo"].isin(concluidos)
].head(100)

tc = np.arange(TAMANHO, dtype=float)
tc -= tc.mean()
den = np.sum(tc ** 2)

permitidos = list(range(10)) + list(range(101, 110))
esperados = aud.set_index("Arquivo")["Janelas_2_ou_mais"]

def todos_validos(vetor, comprimento, inicios):
    """Verifica se todos os elementos da janela sao validos."""
    erros = np.r_[0, np.cumsum(~np.asarray(vetor, dtype=bool))]
    return (
        erros[inicios + comprimento] - erros[inicios]
    ) == 0

sessao_http = requests.Session()

for _, item in lote.iterrows():

    nome = item["Arquivo"]
    print("Processando:", nome)

    try:
        url = (
            "https://raw.githubusercontent.com/petrobras/3W/"
            f"main/dataset/{item['Categoria']}/{nome}"
        )

        resposta = sessao_http.get(url, timeout=180)
        resposta.raise_for_status()
        dados = pd.read_parquet(io.BytesIO(resposta.content))

        sinais = dados.reindex(columns=SENSORES).apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=float)

        rotulos = pd.to_numeric(
            dados["class"], errors="coerce"
        ).to_numpy(dtype=float)

        inicios = np.arange(
            0, len(dados) - TAMANHO + 1, PASSO
        )

        if len(inicios):
            rotulo_ok = (
                todos_validos(
                    np.isin(rotulos, permitidos),
                    TAMANHO, inicios
                )
                &
                todos_validos(
                    rotulos[1:] == rotulos[:-1],
                    TAMANHO - 1, inicios
                )
            )

            delta = np.diff(dados.index.asi8) / 1e9

            tempo_ok = todos_validos(
                np.isclose(delta, 1, atol=0.1),
                TAMANHO - 1, inicios
            )

            completos = np.column_stack([
                todos_validos(
                    np.isfinite(sinais[:, j]),
                    TAMANHO, inicios
                )
                for j in range(len(SENSORES))
            ])

            selecionar = (
                rotulo_ok & tempo_ok
                & (completos.sum(axis=1) >= MIN_SENSORES)
            )

            inicios = inicios[selecionar]
            completos = completos[selecionar]

        n = len(inicios)

        # Verificar se reproduzimos a auditoria da Sessao 10
        assert n == int(esperados.loc[nome]), (
            f"Divergencia de janelas: {nome}"
        )

        if n > 0:
            matriz = pd.DataFrame({
                "Arquivo": nome,
                "Poco": item["Poco"],
                "Categoria_arquivo": int(item["Categoria"]),
                "Rotulo_original": rotulos[inicios].astype(int),
                "Inicio": dados.index[inicios].astype(str),
                "Fim": dados.index[inicios + TAMANHO - 1].astype(str),
                "N_sensores": completos.sum(axis=1)
            })

            for j, sensor in enumerate(SENSORES):
                valido = completos[:, j]

                matriz[f"{sensor}_disponivel"] = valido.astype(int)

                colunas = [
                    "media", "desvio", "inclinacao", "p10", "p90"
                ]

                for atributo in colunas:
                    matriz[f"{sensor}_{atributo}"] = np.nan

                if valido.any():
                    janelas = sliding_window_view(
                        sinais[:, j], TAMANHO
                    )[inicios[valido]]

                    medias = janelas.mean(axis=1)

                    valores = [
                        medias,
                        janelas.std(axis=1),
                        ((janelas - medias[:, None]) @ tc) / den,
                        np.percentile(janelas, 10, axis=1),
                        np.percentile(janelas, 90, axis=1)
                    ]

                    for atributo, valor in zip(colunas, valores):
                        matriz.loc[
                            valido, f"{sensor}_{atributo}"
                        ] = valor

            assert matriz.shape[1] == 37

            destino = pasta_saida / (
                Path(nome).stem + "_atributos.parquet"
            )
            matriz.to_parquet(destino, index=False)

        progresso = pd.concat([
            progresso,
            pd.DataFrame([{
                "Arquivo": nome,
                "Poco": item["Poco"],
                "Janelas": n,
                "Status": "OK"
            }])
        ], ignore_index=True)

        progresso.to_csv(registro, index=False)

    except Exception as erro:
        print("ERRO:", nome, str(erro))
        break

sessao_http.close()

ok = progresso[progresso["Status"] == "OK"]

print("\nPROGRESSO DA SESSAO 11:")
print("Arquivos processados:", len(ok), "/ 535")
print("Pocos processados:", ok["Poco"].nunique())
print("Janelas extraidas:", int(ok["Janelas"].sum()))

print("\nULTIMOS ARQUIVOS:")
display(ok.tail(5))

print("\nRESULTADOS SALVOS EM:")
print(pasta_saida)


Processando: WELL-00002_20170812180000.parquet
Processando: WELL-00002_20170812230000.parquet
Processando: WELL-00002_20170813040000.parquet
Processando: WELL-00002_20170813090000.parquet
Processando: WELL-00002_20170813140000.parquet

PROGRESSO DA SESSAO 11:
Arquivos processados: 535 / 535
Pocos processados: 28
Janelas extraidas: 321929

ULTIMOS ARQUIVOS:


,Arquivo,Poco,Janelas,Status
530,WELL-00002_20170812180000.parquet,WELL-00002,355,OK
531,WELL-00002_20170812230000.parquet,WELL-00002,357,OK
532,WELL-00002_20170813040000.parquet,WELL-00002,358,OK
533,WELL-00002_20170813090000.parquet,WELL-00002,357,OK
534,WELL-00002_20170813140000.parquet,WELL-00002,357,OK



RESULTADOS SALVOS EM:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/11_Matriz_Atributos_V4


In [ ]:

# SESSAO 11 - TAREFA 06
# Conferencia da matriz e distribuicao temporal

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

# Recuperar o registro da extracao
controle = pd.read_csv(
    pasta_saida / "11_Progresso_Extracao.csv"
)

assert len(controle) == 535
assert controle["Arquivo"].is_unique
assert controle["Status"].eq("OK").all()
assert controle["Janelas"].sum() == 321929

rotulos = Counter()
janelas_poco = Counter()
pocos_rotulo = defaultdict(set)

verificados = 0

colunas = (
    ["Arquivo", "Poco", "Rotulo_original", "N_sensores"]
    + [f"{s}_disponivel" for s in SENSORES]
)

# Conferir somente os arquivos que produziram janelas
for _, item in controle[
    controle["Janelas"] > 0
].iterrows():

    caminho = pasta_saida / (
        Path(item["Arquivo"]).stem + "_atributos.parquet"
    )

    assert caminho.exists(), f"Arquivo ausente: {caminho}"

    df = pd.read_parquet(caminho, columns=colunas)

    assert len(df) == int(item["Janelas"])
    assert df["Arquivo"].eq(item["Arquivo"]).all()
    assert df["Poco"].eq(item["Poco"]).all()
    assert df["N_sensores"].ge(2).all()

    # Conferir os indicadores instrumentais
    indicadores = [
        f"{s}_disponivel" for s in SENSORES
    ]

    assert (
        df[indicadores].sum(axis=1) ==
        df["N_sensores"]
    ).all()

    # Contabilizar as janelas por rotulo e poco
    contagem = df["Rotulo_original"].value_counts()

    for rotulo, quantidade in contagem.items():
        rotulos[int(rotulo)] += int(quantidade)
        pocos_rotulo[int(rotulo)].add(item["Poco"])

    janelas_poco[item["Poco"]] += len(df)
    verificados += 1

# Resumo da integridade
print("1. INTEGRIDADE DA MATRIZ:")
print("Arquivos processados:", len(controle))
print("Arquivos com janelas:", verificados)
print("Arquivos sem janelas:",
      int((controle["Janelas"] == 0).sum()))
print("Janelas conferidas:", sum(rotulos.values()))
print("Pocos com janelas:", len(janelas_poco))

assert sum(rotulos.values()) == 321929
assert verificados == 478
assert len(janelas_poco) == 27

# Distribuicao dos rotulos temporais
tabela_rotulos = pd.DataFrame([
    {
        "Rotulo": r,
        "Janelas": rotulos[r],
        "Pocos": len(pocos_rotulo[r])
    }
    for r in sorted(rotulos)
])

tabela_rotulos["Percentual"] = (
    tabela_rotulos["Janelas"] /
    321929 * 100
).round(2)

print("\n2. DISTRIBUICAO DOS ROTULOS TEMPORAIS:")
display(tabela_rotulos)

# Distribuicao por poco
tabela_pocos = pd.DataFrame([
    {"Poco": p, "Janelas": n}
    for p, n in janelas_poco.items()
]).sort_values("Janelas", ascending=False)

tabela_pocos["Percentual"] = (
    tabela_pocos["Janelas"] /
    321929 * 100
).round(2)

print("\n3. DEZ POCOS COM MAIS JANELAS:")
display(tabela_pocos.head(10))

# Guardar os resumos para o registro e o artigo
tabela_rotulos.to_csv(
    pasta_saida / "11_Distribuicao_Rotulos_V4.csv",
    index=False
)

tabela_pocos.to_csv(
    pasta_saida / "11_Distribuicao_Pocos_V4.csv",
    index=False
)

print("\nTAREFA 06 EXECUTADA")


1. INTEGRIDADE DA MATRIZ:
Arquivos processados: 535
Arquivos com janelas: 478
Arquivos sem janelas: 57
Janelas conferidas: 321929
Pocos com janelas: 27

2. DISTRIBUICAO DOS ROTULOS TEMPORAIS:


,Rotulo,Janelas,Pocos,Percentual
0,0,138404,27,42.99
1,1,58,1,0.02
2,2,299,5,0.09
3,3,10967,1,3.41
4,4,20211,3,6.28
5,5,68,1,0.02
6,6,248,1,0.08
7,7,34,1,0.01
8,8,1856,7,0.58
9,9,309,2,0.10



3. DEZ POCOS COM MAIS JANELAS:


,Poco,Janelas,Percentual
0,WELL-00002,89738,27.88
9,WELL-00022,61456,19.09
10,WELL-00023,30913,9.60
12,WELL-00026,25682,7.98
25,WELL-00041,19563,6.08
6,WELL-00014,16293,5.06
11,WELL-00025,14366,4.46
13,WELL-00027,13531,4.20
7,WELL-00020,7179,2.23
16,WELL-00031,7171,2.23



TAREFA 06 EXECUTADA
